# Evaluatie Functie 2: Classify-then-Retrieve — Classificatieaccuracy

**Methode:** de classifier (`classificeer_vraag`) ontvangt een werknemersvraag en kiest 1–3 beleidslabels uit een vaste lijst van 13 categorieën, of geeft `GEEN_MATCH` terug als de vraag buiten het mobiliteitsbeleid valt.  
In deze evaluatie vergelijken we het **eerste gekozen label** (het meest relevante) met het verwachte label uit de testset.  
Metriek: **accuracy** (aantal correct gekozen labels / totaal aantal vragen).

## 1. Imports & configuratie

In [ ]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

# Detecteer de projectmap (notebook staat in evaluatie/)
_cwd = Path.cwd()
PROJECT_DIR  = _cwd.parent if _cwd.name == "evaluatie" else _cwd
EVAL_DIR     = PROJECT_DIR / "evaluatie"
DATASET_PATH = EVAL_DIR / "eval_dataset_f2.json"
RESULTATEN_DIR = EVAL_DIR / "resultaten"
RESULTATEN_DIR.mkdir(exist_ok=True)

print(f"PROJECT_DIR  : {PROJECT_DIR}")
print(f"DATASET_PATH : {DATASET_PATH}")

load_dotenv(PROJECT_DIR / ".env")

# Voeg src/ toe aan het zoekpad zodat classifier.py importeerbaar is
sys.path.insert(0, str(PROJECT_DIR / "src"))
from classifier import classificeer_vraag

print("\nAlles geladen ✓")

## 2. Testset inladen

In [ ]:
with open(DATASET_PATH, encoding="utf-8") as f:
    testset = json.load(f)

print(f"Testset geladen: {len(testset)} vragen")

# Overzicht van de labels in de testset
labels_verwacht = [v["verwacht_label"] for v in testset]
unieke_labels = sorted(set(labels_verwacht))
print(f"Unieke verwachte labels ({len(unieke_labels)}): {unieke_labels}")

## 3. Aanpak

Per vraag:
1. Roep `classificeer_vraag(vraag)` aan — dit stuurt de vraag naar de LLM-classifier.
2. Neem het **eerste teruggegeven label** als het primaire antwoord van de classifier.
3. Vergelijk dit label met het verwachte label uit de testset.
4. Sla het resultaat op als `CORRECT` of `FOUT`.

> **Let op:** de classifier kan meerdere labels teruggeven. We evalueren alleen op het eerste label om het eenvoudig en eenduidig te houden.

## 4. Evaluatieloop

In [ ]:
resultaten = []

print()
print("=" * 60)
print("EVALUATIE FUNCTIE 2 — Classificatie")
print("=" * 60)

for entry in testset:
    idx            = entry["id"]
    vraag          = entry["vraag"]
    verwacht_label = entry["verwacht_label"]

    try:
        gekozen_labels = classificeer_vraag(vraag)
        gekozen_label  = gekozen_labels[0] if gekozen_labels else "GEEN_MATCH"
    except Exception as e:
        gekozen_label = f"FOUT: {e}"

    is_correct = gekozen_label == verwacht_label
    status     = "CORRECT" if is_correct else "FOUT"

    resultaten.append({
        "id"             : idx,
        "vraag"          : vraag,
        "verwacht_label" : verwacht_label,
        "gekozen_label"  : gekozen_label,
        "correct"        : is_correct
    })

    print(f"\n[{idx}] \"{vraag[:70]}{'...' if len(vraag) > 70 else ''}\"")
    print(f"    Verwacht: {verwacht_label:<35} | Gekozen: {gekozen_label:<35} | {status}")

print()
print("-" * 60)

## 5. Accuracy berekenen

In [ ]:
totaal  = len(resultaten)
correct = sum(1 for r in resultaten if r["correct"])
accuracy = (correct / totaal) * 100

print(f"Accuracy: {accuracy:.1f}% ({correct}/{totaal})")
print("=" * 60)

## 6. Confusion matrix

De confusion matrix toont voor elk **verwacht label** (rijen) welk label de classifier daadwerkelijk koos (kolommen).  
- **Diagonaal** (links-boven naar rechts-onder): correct geclassificeerde vragen.  
- **Buiten de diagonaal**: fouten — welk label werd verward met welk ander label.

Zo zie je in één oogopslag of de classifier specifieke labels stelselmatig verwart.

In [ ]:
import matplotlib
matplotlib.use("Agg")           # backend zonder venster, zodat opslaan altijd werkt
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = [r["verwacht_label"] for r in resultaten]
y_pred = [r["gekozen_label"]  for r in resultaten]

# Alle labels die voorkomen (verwacht of gekozen), gesorteerd
alle_labels = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=alle_labels)

fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=alle_labels)
disp.plot(
    ax=ax,
    colorbar=False,
    xticks_rotation=45,
    cmap="Blues"
)
ax.set_title("Confusion Matrix — Functie 2 (Classificatie)", fontsize=14, pad=16)
ax.set_xlabel("Gekozen label (classifier)", fontsize=11)
ax.set_ylabel("Verwacht label (testset)", fontsize=11)
plt.tight_layout()

# Opslaan als PNG
uitvoer_pad = RESULTATEN_DIR / "f2_confusion_matrix.png"
fig.savefig(uitvoer_pad, dpi=150, bbox_inches="tight")
print(f"Confusion matrix opgeslagen: {uitvoer_pad}")

# Toon ook inline in het notebook
matplotlib.use("inline")        # terug naar inline voor weergave
plt.show()
print(f"\nAccuracy: {accuracy:.1f}% ({correct}/{totaal})")
print("=" * 60)

## 7. Resultaten opslaan

In [ ]:
samenvatting = {
    "totaal"   : totaal,
    "correct"  : correct,
    "fout"     : totaal - correct,
    "accuracy" : round(accuracy, 2),
    "resultaten": resultaten
}

json_pad = RESULTATEN_DIR / "f2_resultaten.json"
with open(json_pad, "w", encoding="utf-8") as f:
    json.dump(samenvatting, f, ensure_ascii=False, indent=2)

print(f"Resultaten opgeslagen: {json_pad}")